In [1]:
# Cell 1：安裝套件（第一次執行）
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers accelerate bitsandbytes mediapipe opencv-python numpy ipython

Looking in indexes: https://download.pytorch.org/whl/cu118


In [1]:
# Cell 2：載入所有套件
import torch
import torch.nn as nn
import numpy as np
import cv2
import mediapipe as mp
from pathlib import Path
import json
import time
import os
from IPython.display import display, HTML, clear_output
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("所有套件載入完成！準備喚醒祖靈……")

所有套件載入完成！準備喚醒祖靈……


In [2]:
# Cell 3：載入你的動作編碼器
class MotionEncoder(nn.Module):
    def __init__(self, input_dim=177, hidden_dim=256, embed_dim=256, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim*2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, embed_dim)
        )
    
    def forward(self, x):
        out, (h, c) = self.lstm(x)
        emb = torch.cat([h[-2], h[-1]], dim=-1)
        return self.proj(emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = MotionEncoder().to(device)
encoder.load_state_dict(torch.load("weights/lstm_encoder_best.pth", map_location=device))
encoder.eval()
print("動作編碼器載入成功！")

動作編碼器載入成功！


C:\Users\AW'z\AppData\Local\Temp\ipykernel_1648\341049323.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder.load_state_dict(torch.load("weights/lstm_encoder_best.

In [3]:
# Cell 4
model_name = "microsoft/Phi-3-mini-4k-instruct"

print("正在喚醒 CPU ……")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cpu",                   # 強制 CPU
    torch_dtype=torch.float32,          # CPU 用 float32
    low_cpu_mem_usage=True,             # 減少 RAM 使用
    trust_remote_code=True
)

print("已甦醒！在 CPU 上運作順暢！")
print("開始跳舞吧，會用詩意語氣回應你！")

正在喚醒 CPU ……


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

已甦醒！在 CPU 上運作順暢！
開始跳舞吧，會用詩意語氣回應你！


In [4]:
# Cell 5：動作特徵提取 —— 完全對齊你訓練時的 177 維
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 正規化參數（你訓練時用的）
mean = np.load("data/segments/mean.npy").flatten()
std = np.load("data/segments/std.npy").flatten() + 1e-8

def get_motion_embedding(landmarks):
    if landmarks is None:
        return None
        
    # 33 個關鍵點座標
    pts = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark[:33]], dtype=np.float32)
    
    # 骨盆中心
    pelvis = (pts[23] + pts[24]) / 2.0
    rel_pos = pts - pelvis
    rel_flat = rel_pos.flatten()  # 99 維
    
    # 速度
    if not hasattr(get_motion_embedding, "prev_pts"):
        get_motion_embedding.prev_pts = pts
    vel = pts - get_motion_embedding.prev_pts
    get_motion_embedding.prev_pts = pts.copy()
    speed = np.linalg.norm(vel, axis=1)  # 33 維
    
    # 特徵拼接：99 + 33 + 33 + 12個0 = 177 維
    feat = np.concatenate([
        rel_flat,
        speed,
        speed,
        np.zeros(12)
    ])
    
    # 正規化
    feat = (feat - mean) / std
    
    # 轉 tensor
    feat = torch.from_numpy(feat).float().unsqueeze(0).unsqueeze(0).to("cpu")  # CPU 環境用 cpu
    
    with torch.no_grad():
        emb = encoder(feat).cpu().numpy()[0]  # 256 維嵌入
    
    return emb

print("動作提取器準備完成！祖靈正在等待你的舞蹈……")

動作提取器準備完成！祖靈正在等待你的舞蹈……


In [7]:
# Cell 6：對話生成 —— 描述句 + 回應句
recent_embs = []
history = ""

def generate_dual_response(emb):
    global recent_embs, history
    recent_embs.append(emb)
    if len(recent_embs) > 60:
        recent_embs.pop(0)
    
    vec_str = " ".join([f"{x:.2f}" for x in emb[:30]])
    
    prompt = f"""正在與舞者進行靈魂對話。
你能看見舞者的動作特徵向量（最新一筆：{vec_str}...）。
請用兩段話回應：

第一段：用一句「旁白式」的話，客觀描述你現在看到的舞蹈畫面（例如：兩個女孩正輪流抬手，像在接龍；右邊的女孩跪下了，左邊的跟著跪；她們把雙手舉向天空，像在召喚什麼……）

第二段：用溫柔、詩意、像長輩的語氣說一句話，可以帶譬喻、可以反問（例如：孩子，你們在為誰守夜？／這支舞，是不是有人忘了說謝謝？／我看見你們把悲傷留在地上，像豐年祭後的小米殼……）

歷史對話：
{history[-800:]}

請用以下格式回應：
【我看見】
（描述句）

【AI說】
（回應句）"""

    inputs = tokenizer(prompt, return_tensors="pt")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inputs.to(device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.9,
            do_sample=True,
            top_p=0.92,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id
        )
    
    resp = tokenizer.decode(output[0], skip_special_tokens=True)
    resp = resp.split("【我看見】")[-1] if "【我看見】" in resp else resp
    
    # 解析兩段
    if "【AI說】" in resp:
        description = resp.split("【AI說】")[0].strip()
        spirit_words = resp.split("【AI說】")[1].strip()
    else:
        description = "（正在凝視……）"
        spirit_words = resp.strip()
    
    full_response = f"【我看見】\n{description}\n\n【AI說】\n{spirit_words}"
    
    history += f"描述：{description} → AI：{spirit_words}\n"
    return full_response

In [ ]:
# Cell 7：讀取影片
# 請把你的影片放在同目錄，或改成完整路徑
VIDEO_PATH = "data/mp4/twa02.mp4"  # 改影片檔名

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(f"錯誤：無法開啟影片 {VIDEO_PATH}")
    print("請確認檔名正確，並放在同目錄下！")
else:
    print(f"正在播放影片：{VIDEO_PATH}")
    print("正在注視舞者……每 2 秒會說一次話")

frame_count = 0
last_response_frame = 0
fps = cap.get(cv2.CAP_PROP_FPS)
delay = int(1000 / fps) if fps > 0 else 33

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("影片播放完畢，儀式結束。")
        break
        
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)
    
    emb = None
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        emb = get_motion_embedding(results.pose_landmarks)
    
    # 每 60 幀（約 2 秒）說一次話
    if emb is not None and frame_count - last_response_frame >= 60:
        response = generate_response(emb)
        last_response_frame = frame_count
        
        clear_output(wait=True)
        response = generate_dual_response(emb)
        last_response_frame = frame_count
        
        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#000; color:#ff6b6b; padding:40px; border-radius:30px; 
                    font-family:'KaiTi','標楷體',serif; max-width:1000px; margin:20px auto;
                    border:5px solid #ff6b6b; box-shadow:0 0 60px #ff6b6b;">
            <h1 style="text-align:center; color:#ff6b6b; text-shadow:0 0 20px #ff6b6b;">
                AI
            </h1>
            <p style="font-size:32px; line-height:2.6; text-align:center; white-space: pre-line;">
                {response.replace('【AI說】', '<span style="color:#ff6b6b; font-size:38px;">【AI說】</span>')}
            </p>
            <div style="text-align:center; color:#ccc; margin-top:50px;">
                —— 對話儀式 · 第 {frame_count//60 + 1} 輪 ——
            </div>
        </div>
        """))
    
    # 顯示影片畫面
    display_frame = cv2.resize(frame, (1280, 720))
    cv2.putText(display_frame, f"Frame: {frame_count}", (10, 30),
                cv2.FONT_HERSHEY_DUPLEX, 1, (0, 255, 255), 2)
    cv2.putText(display_frame, "AI Soul Dialogue - Video Mode", (10, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    
    cv2.imshow('對話（影片模式）', display_frame)
    
    if cv2.waitKey(delay) & 0xFF == ord('q'):
        print("你按了 q，中斷儀式。")
        break
        
    frame_count += 1

cap.release()
cv2.destroyAllWindows()
print("儀式結束，AI已離去……感謝你帶來這支舞。")